# 🌩️ Alerta de tormentas severas — Servicio Meteorológico Nacional (ficticio)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ProyectoIngenieria-fiuba/PIProgramando/blob/main/sesion_3/alerta_tormentas_smn.ipynb)

> **Aviso:** todos los datos son **ficticios**, generados para este taller. No tienen ninguna
> relación con los sistemas, modelos ni datos reales del SMN.

## La situación

Son el equipo de ciencia de datos del **Servicio Meteorológico** (ficticio). El equipo anterior
dejó este notebook antes de irse. Dice que el modelo de **alerta de tormenta severa** ya está listo y
que el lunes lo ponen a emitir alertas de verdad para cinco ciudades.

Les piden una sola cosa: **antes del lunes, ¿podemos confiar en este modelo?**

## Cómo se trabaja en este notebook

1. **Primero corran todo** de arriba hacia abajo, sin cambiar nada (hasta la sección *Notebook heredado*).
2. Después, trabajen en las secciones que siguen. Pueden usar la IA que quieran, pero con una regla:
   **antes de aplicar cualquier cambio** (lo haya propuesto la IA o ustedes) completan un
   **checkpoint**: qué métrica esperan que se mueva y por qué. Después de correrlo, anotan qué pasó.
3. Los facilitadores van a pasar por la sala y les van a preguntar:
   **«¿cómo saben que esto está bien?»**
4. Todo lo que anoten en los checkpoints se convierte, al final, en su *prompt log* (sección 5).

Material de apoyo: `CONSIGNA.md`, `TEORIA.md` y `GUIA_IA.md` (están en la misma carpeta del repo).

## 0. Preparación

Las dos celdas siguientes dejan listas las librerías y las herramientas del registro de checkpoints.
**No hace falta tocarlas** (si quieren, léanlas: son cortas).

In [ ]:
#@title Librerías y herramientas del registro de checkpoints { display-mode: "form" }
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# ---- Registro de checkpoints (alimenta el prompt log) ---------------------------------
REGISTRO = []

def checkpoint(pedido, espero, porque):
    """ANTES de aplicar un cambio: qué pidieron, qué métrica esperan que se mueva y por qué."""
    campos = {"pedido": pedido, "espero": espero, "porque": porque}
    vacios = [k for k, v in campos.items() if len(v.strip()) < 15]
    if vacios:
        print("⛔ Checkpoint incompleto. Completen bien estos campos antes de aplicar el cambio:", vacios)
        return
    REGISTRO.append({**campos, "paso": "", "verifique": ""})
    print(f"✅ Checkpoint #{len(REGISTRO)} anotado. Ahora sí: apliquen el cambio y corran.")

def cerrar_checkpoint(n, paso, verifique):
    """DESPUÉS de correr: qué pasó realmente y cómo lo verificaron (con números, no con 'se ve bien')."""
    if not (1 <= n <= len(REGISTRO)):
        print("⛔ Ese checkpoint no existe. Primero anótenlo con checkpoint(...).")
        return
    if len(paso.strip()) < 15 or len(verifique.strip()) < 15:
        print("⛔ Contesten 'paso' y 'verifique' con más detalle (qué número se movió, cuánto, y cómo lo comprobaron).")
        return
    REGISTRO[n - 1].update(paso=paso, verifique=verifique)
    print(f"✅ Checkpoint #{n} cerrado.")

def ver_registro():
    """Muestra una tabla con todos los checkpoints."""
    pd.set_option("display.max_colwidth", None)
    return pd.DataFrame(REGISTRO, index=range(1, len(REGISTRO) + 1))

def prompt_log_markdown():
    """Devuelve el prompt log como tabla markdown, lista para pegar en el README."""
    filas = ["| # | Qué pedí | Qué esperaba ver mover y por qué | Qué pasó | Cómo verifiqué |",
             "|---|----------|----------------------------------|----------|----------------|"]
    for i, r in enumerate(REGISTRO, 1):
        celdas = [r["pedido"], f'{r["espero"]} — {r["porque"]}', r["paso"], r["verifique"]]
        filas.append(f"| {i} | " + " | ".join(c.replace("|", "/").replace("\n", " ") for c in celdas) + " |")
    return "\n".join(filas)

### Los datos

Cada fila es **una observación de una estación meteorológica cada 3 horas**, durante dos años
(julio 2023 – junio 2025), para cinco estaciones. La columna `tormenta_severa` vale **1** si en las
3 horas que arrancan en `fecha_hora` hubo una tormenta severa en la zona de la estación, y **0** si no.

La alerta se emite **en el momento `fecha_hora`**, mirando lo que las estaciones midieron hasta ese instante.

| Columna | Qué es |
|---|---|
| `fecha_hora` | Momento de la observación (UTC-3) |
| `estacion` | Estación: Aeroparque, Córdoba, Mendoza, Resistencia, Tucumán |
| `temperatura_c` | Temperatura del aire (°C) |
| `punto_rocio_c` | Temperatura de punto de rocío (°C) |
| `humedad_pct` | Humedad relativa (%) |
| `presion_hpa` | Presión reducida a nivel del mar (hPa) |
| `var_presion_3h` | Cambio de presión respecto de la observación anterior, 3 h antes (hPa) |
| `viento_kmh` | Velocidad del viento (km/h) |
| `dir_viento` | Dirección desde la que sopla el viento (N, NE, E, SE, S, SO, O, NO) |
| `nubosidad_octas` | Cobertura de nubes, en octavos de cielo (0 = despejado, 8 = cubierto) |
| `indice_inestabilidad` | Energía disponible para convección (J/kg), calculada con el último globo sonda |
| `lluvia_previa_mm` | Lluvia que cayó en las 3 h anteriores a la observación (mm) |
| `precip_acum_mm` | Precipitación acumulada (mm) del registro del pluviómetro para el bloque de 3 h que arranca en `fecha_hora` |
| `tormenta_severa` | **Lo que queremos predecir.** 1 = hubo tormenta severa, 0 = no |

In [ ]:
#@title Carga de los datos (embebidos en el notebook, no hace falta tocar esta celda) { display-mode: "form" }
import base64, gzip, io
_DATOS = "..."
df = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(_DATOS))), parse_dates=["fecha_hora"])
print(f"{len(df):,} observaciones cargadas.")

---
## Notebook heredado

> Todo lo de esta sección lo escribió el equipo anterior. **Corranlo sin modificarlo.**

In [ ]:
df.head()

In [ ]:
# Variables de entrada (X) y lo que queremos predecir (y)
df["mes"] = df["fecha_hora"].dt.month
df["hora"] = df["fecha_hora"].dt.hour
y = df["tormenta_severa"]
X = pd.get_dummies(df.drop(columns=["fecha_hora", "tormenta_severa"]),
                   columns=["estacion", "dir_viento"], dtype=int)

# Separamos por fecha: el modelo siempre predice el futuro con datos del pasado
FECHA_CORTE = "2025-01-01"
es_train = df["fecha_hora"] < FECHA_CORTE
X_train, X_test = X[es_train], X[~es_train]
y_train, y_test = y[es_train], y[~es_train]
print(f"Entrenamiento: {len(X_train):,} filas  |  Prueba: {len(X_test):,} filas")

In [ ]:
# Entrenamos el modelo
modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modelo.fit(X_train, y_train)

In [ ]:
# Evaluamos en los datos de prueba
pred = modelo.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, pred):.2%}")

> 📝 *Nota del equipo anterior:* **«¡98% de accuracy! El modelo anda perfecto. Listo para producción el lunes. 🚀»**

---
## 1. Diagnóstico — ¿le creemos a ese 98%?

Antes de tocar nada, **investiguen**. Algunas preguntas para arrancar (no son las únicas):

- ¿Qué proporción de las observaciones son tormentas severas? ¿Y en el conjunto de prueba?
- ¿Qué accuracy tendría un «modelo» que siempre responde *«no hay tormenta»*?
- ¿Qué métricas necesitamos para saber si el modelo **detecta tormentas**? ¿Cuánto valen?

Usen celdas nuevas (`+ Código`) para explorar. Pueden pedirle ayuda a la IA, pero ojo con
`GUIA_IA.md`: la IA no conoce sus datos.

In [ ]:
# Exploren acá (agreguen todas las celdas que necesiten)

### 🛑 Checkpoint de diagnóstico

Antes de pasar a corregir, cada cambio o análisis nuevo se anota. Pueden usar este formato:

In [ ]:
checkpoint(
    pedido="",     # qué le pidieron a la IA (o 'lo hicimos a mano: ...')
    espero="",     # qué métrica esperan ver y qué valor
    porque="",     # por qué lo esperan
)

---
## 2. Corrección (objetivo central)

Corrijan lo que haga falta hasta tener un modelo **en el que se pueda confiar**, y evalúenlo con
las métricas que corresponden. Trabajen por iteraciones chicas: **un cambio por vez**, así saben qué
produjo cada movimiento de las métricas.

Cada iteración tiene tres pasos:

1. **Checkpoint (antes):** anotan qué pidieron / qué van a cambiar, qué métrica esperan que se mueva y por qué.
2. **Cambio:** lo aplican y corren.
3. **Cierre (después):** anotan qué pasó realmente y **cómo lo verificaron**.

Si necesitan más iteraciones, copien y peguen las celdas.

### Iteración 1

In [ ]:
checkpoint(
    pedido="",
    espero="",
    porque="",
)

In [ ]:
# Cambio / código de la iteración 1


# Cierre (cuando ya corrieron y miraron los números):
# cerrar_checkpoint(n=..., paso="", verifique="")

### Iteración 2

In [ ]:
checkpoint(
    pedido="",
    espero="",
    porque="",
)

In [ ]:
# Cambio / código de la iteración 2


# Cierre (cuando ya corrieron y miraron los números):
# cerrar_checkpoint(n=..., paso="", verifique="")

### Iteración 3

In [ ]:
checkpoint(
    pedido="",
    espero="",
    porque="",
)

In [ ]:
# Cambio / código de la iteración 3


# Cierre (cuando ya corrieron y miraron los números):
# cerrar_checkpoint(n=..., paso="", verifique="")

### ✍️ Conclusión del objetivo central

Escriban acá, **con sus propias palabras** (sin pegar lo que dijo la IA):

1. **¿Por qué el 98% inicial era falso?** (Una explicación que le puedan dar a alguien que no sabe de ML).
2. **¿Qué números tiene ahora el modelo?** (las métricas que correspondan, en el conjunto de prueba)
3. **¿Cómo saben que esos números ahora sí son confiables?**

> *(escriban acá)*

---
## 3. Extras (en orden — hagan los que alcancen)

Los extras valen igual que el objetivo central en **método**: siguen con checkpoint antes y cierre después.

### Extra 1 — El umbral de alerta

Hoy el modelo alerta cuando su probabilidad pasa 0.5. Pero **no es lo mismo** equivocarse para un lado que
para el otro: una **falsa alarma** cuesta credibilidad, evacuaciones y plata; una **tormenta no detectada**
puede costar vidas. Decidan un umbral, **con un criterio que puedan explicar en términos del SMN**, y
justifíquenlo.

Pista de método: el umbral se elige con datos que **no sean** los de prueba.

In [ ]:
checkpoint(
    pedido="",
    espero="",
    porque="",
)

In [ ]:
# código del extra


# cerrar_checkpoint(n=..., paso="", verifique="")

### Extra 2 — Un segundo modelo

Prueben otro tipo de modelo (por ejemplo uno basado en árboles) y compárenlo con el primero **con las
mismas métricas y el mismo umbral de decisión**. ¿Es mejor? ¿En qué? ¿Vale la pena la complejidad?

In [ ]:
checkpoint(
    pedido="",
    espero="",
    porque="",
)

In [ ]:
# código del extra


# cerrar_checkpoint(n=..., paso="", verifique="")

### Extra 3 — ¿Qué mira el modelo?

Calculen la **importancia de las variables** y expliquen **en lenguaje llano** qué es lo que mira el modelo
para decidir. ¿Tiene sentido meteorológico? ¿Hay algo que les llame la atención?

In [ ]:
checkpoint(
    pedido="",
    espero="",
    porque="",
)

In [ ]:
# código del extra


# cerrar_checkpoint(n=..., paso="", verifique="")

---
## 4. Documentación — README + prompt log

Esto es lo que van a poder mostrar después (portafolio, CV, entrevistas): **un proyecto con su proceso documentado**.

Corran `ver_registro()` para ver sus checkpoints y `print(prompt_log_markdown())` para obtener la tabla del
prompt log lista para pegar en el README.

In [ ]:
ver_registro()

In [ ]:
print(prompt_log_markdown())

### Plantilla del README

Copien esto a un archivo `README.md` y completen:

```markdown
# Alerta de tormentas severas — auditoría de un modelo heredado

## Resumen
*(2-3 líneas: qué era el proyecto y qué encontraron)*

## Qué estaba mal en el modelo original
*(con sus palabras; para cada problema: qué era, cómo lo detectaron, cómo lo arreglaron)*

## Resultados
| Modelo | Recall | Precisión | F1 | AUC-PR |
|---|---|---|---|---|
| Heredado | | | | |
| Corregido | | | | |

*(¿Con qué conjunto de datos se midió? ¿Qué umbral se usó y por qué?)*

## Qué mira el modelo
*(en lenguaje llano)*

## Limitaciones y próximos pasos
*(datos sintéticos, qué no se probó, qué harían con más tiempo)*

## Prompt log
*(pegar acá la tabla de `prompt_log_markdown()`)*
```